# scSVC-reconstructed CAFs uncovers their functionally distinct niches within the tumor microenvironment

In [ ]:
output_dir = "../../output/sc_SVC_case/P2CRC_Xenium"
select_ct = "Fibroblast"


In [ ]:
import os
import scanpy as sc

from revise.application.sc_svc import ScSVCAnalysis

svc_save_dir = f"{output_dir}/{select_ct}"
sc_svc_expr = sc.read_h5ad(f"{svc_save_dir}/sc_SVC_expr.h5ad")
sc_svc_spatial = sc.read_h5ad(f"{svc_save_dir}/sc_SVC_spatial.h5ad")

sc_svc_analysis = ScSVCAnalysis(sc_svc_spatial, sc_svc_expr, 
                            "SVC_cluster")

In [ ]:
cm_df = sc_svc_analysis.get_cm_df("Level2")
cm_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors 

size = None
cmap = plt.cm.get_cmap('tab20', lut=10)
palette = [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]

sc.pl.scatter(sc_svc_analysis.sc_SVC_adata_spatial, x="x", y="y",
              color='Level2',
              size=size)
desired_order = ['0', '1', '5', '3', '4', '2', '6', '7', '8', '9']  # for better visualization
sc_svc_analysis.sc_SVC_adata_spatial.obs['SVC_cluster'] = (
    sc_svc_analysis.sc_SVC_adata_spatial.obs['SVC_cluster'].cat.reorder_categories(desired_order, ordered=True)
)
sc.pl.scatter(sc_svc_analysis.sc_SVC_adata_spatial, x="x", y="y",
              color='SVC_cluster',
              size=size)

In [ ]:
import matplotlib.pyplot as plt
def plot_sc_SVC(adata, color, title = None, file_name = None):

    plt.figure(figsize=(10, 8*len(color)))
    sc.pl.scatter(
        adata, x="x", y="y",
        color = color,
        title=title, show = False,
            )
    plt.savefig(file_name, dpi = 300)
    plt.close()

sc_SVC_file_name = f"{output_dir}/sc_SVC.png"
plot_sc_SVC(sc_svc_analysis.sc_SVC_adata_spatial, color='SVC_cluster',
            file_name=sc_SVC_file_name
            )

sc_SVC_file_name = f"{output_dir}/compare.png"
plot_sc_SVC(sc_svc_analysis.sc_SVC_adata_spatial, color=['Level2','SVC_cluster'],
            title=["Expert anno", "sc_SVC"],
            file_name=sc_SVC_file_name
            )

## Marker gene analysis

In [ ]:
sc_svc_analysis.sc_SVC_degs.to_csv(f"{output_dir}/degs_all.csv")
sc_svc_analysis.sc_SVC_degs

In [ ]:
cluster_nums = ['1', '4', '5', '6', '8']

degs = sc_svc_analysis.get_svc_degs(cluster_nums)
marker_dict = (
    degs.groupby('group')['gene']
    .apply(lambda x: x.head(10).tolist())
    .to_dict()
)
sc_svc_analysis.get_dot_plot(cluster_nums, marker_dict)

In [ ]:
for i,j in marker_dict.items():
    print(i, j)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.useafm'] = False

marker_dict = {
    "1": ['COMP', 'TIMP3', 'COL11A1'], # ECM-Architect	
    "4": ['MMP11',  # Inflamed-Proteolytic
          'MMP14', 'HTRA3'],
    "5": ['TAGLN', 'ACTA2', 'GREM1', #  SMC-like CAF	
          ],
    "8": ['MMP1', 'MMP3', 'PLAU', # ECM-Destructive	desCAF	
          ]
    }

sc_svc_analysis.get_dot_plot(cluster_nums, marker_dict, normalize=True)

In [ ]:
plt.figure(figsize=(8, 6))
sc_svc_analysis.get_dot_plot(cluster_nums, marker_dict, normalize=True)
plt.savefig(f"{output_dir}/sc_SVC_dotplot.pdf", dpi=300, bbox_inches='tight')
plt.close()

## sc_SVC subclusters bioinformatical analysis

In [ ]:
cluster_nums

In [ ]:
sc_svc_analysis.get_violin_plot(cluster_nums, True)

In [ ]:
fc_threshold = 1
pathway_num = 20
gene_num = 60
geneset_file = ["MSigDB_Hallmark_2020", "KEGG_2021_Human", "GO_Biological_Process_2025"]
# geneset_file = ["MSigDB_Hallmark_2020"]
pathway_file_name = f"{output_dir}/pathway_{fc_threshold}_{pathway_num}.txt"
all_pathway = sc_svc_analysis.get_pathway_conclusion(
    cluster_nums, fc_threshold=fc_threshold, pathway_num=pathway_num, gene_num=gene_num, geneset_file=geneset_file, normalize=True)
all_pathway.to_csv(pathway_file_name)

In [ ]:
caf_cluster_num = '1' # Arch，
caf_cluster_num = '4' # Inflamed
caf_cluster_num = '5' # SMC-like
caf_cluster_num = '8' # Destructive

# for enrichment analysis
os.makedirs(f'{output_dir}/pathway', exist_ok = True)
from revise.tools.bio import pathway_barplot, pathway_network_plot
for caf_cluster_num in ['1', '4', '5', '8']:
    pathway = all_pathway[all_pathway['group'] == caf_cluster_num]

    pathway_barplot(pathway, pathway_num = 10,  
                    save_file_name = f'{output_dir}/pathway/{caf_cluster_num}_barplot.pdf')
    pathway_network_plot(pathway, 
                        top_term = 6, 
                        save_file_name = f'{output_dir}/pathway/{caf_cluster_num}_network.pdf')

In [ ]:
top_pathway = all_pathway.groupby('group').head(10)

top_pathway.reset_index(drop=True, inplace=True)
for caf_cluster_num in top_pathway['group'].unique():
    pathway = top_pathway[top_pathway['group'] == caf_cluster_num]
    print(caf_cluster_num, pathway['Term'].values)


In [ ]:
pathway_dict = {
    "1": ['Epithelial Mesenchymal Transition', 'Protein digestion and absorption', 'ECM-receptor interaction', 'Focal adhesion'],
    "4": ['Positive Regulation of Blood Coagulation (GO:0030194)','Complement and coagulation cascades', 'IL-2/STAT5 Signaling', 'Interferon Alpha Response'],
    "5": ['Vascular smooth muscle contraction', 'Muscle Contraction (GO:0006936)', 'Actomyosin Structure Organization (GO:0031032)', 'Regulation of actin cytoskeleton'],
    "8": ['External Encapsulating Structure Organization (GO:0045229)',
         'TNF-alpha Signaling via NF-kB', 'TNF signaling pathway', 'IL-17 signaling pathway'],
}


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42      
mpl.rcParams['ps.useafm'] = False      

def pathway_barplot(all_pathway, pathway_dict=None, select_metric='Adjusted P-value', cmap='RdBu_r', save_file_name=None):
    df = all_pathway.copy()
    
    if select_metric == 'Adjusted P-value':
        df["plot_value"] = -np.log10(df[select_metric])
        xlabel = '-log10(Adjusted P-value)'
    else:
        df["plot_value"] = df[select_metric]
        xlabel = select_metric
    
    if pathway_dict is None:
        pathway_dict = {}
        unique_groups = df['group'].unique()
        for group in unique_groups:
            group_pathways = df[df['group'] == group]['Term'].tolist()
            pathway_dict[group] = group_pathways
    
    all_groups = list(pathway_dict.keys())
    all_pathways = []
    for pathways in pathway_dict.values():
        all_pathways.extend(pathways)
    all_pathways = list(set(all_pathways))
    
    data_matrix = {}
    for pathway in all_pathways:
        data_matrix[pathway] = {}
        for group in all_groups:
            mask = (df['Term'] == pathway) & (df['group'] == group)
            if mask.any():
                data_matrix[pathway][group] = df[mask]['plot_value'].iloc[0]
            else:
                data_matrix[pathway][group] = 0  # 不存在则为0
    
    if isinstance(cmap, dict):
        colors_dict = cmap
        def get_color(group):
            return colors_dict.get(str(group), '#808080')  # 默认灰色
    else:
        colors = plt.cm.get_cmap(cmap, len(all_groups))
        def get_color(group):
            color_idx = all_groups.index(group)
            return colors(color_idx)
    
    n_subplots = len(pathway_dict)
    fig, axes = plt.subplots(1, n_subplots, figsize=(12, 8))
    if n_subplots == 1:
        axes = [axes]
    
    for idx, (group_name, pathways) in enumerate(pathway_dict.items()):
        ax = axes[idx]
        
        bar_data = []
        bar_labels = []
        pathway_positions = []
        
        current_pos = 0
        group_spacing = 0.8  
        
        pathways = list(reversed(pathways))
        
        reversed_groups = list(reversed(all_groups))
        
        for pathway_idx, pathway in enumerate(pathways):
            pathway_start_pos = current_pos
            
            for group_idx, group in enumerate(reversed_groups):
                value = data_matrix[pathway][group]
                bar_data.append({
                    'position': current_pos,
                    'value': value,
                    'group': group,
                    'pathway': pathway
                })
                bar_labels.append(group)
                current_pos += group_spacing
            
            pathway_center = pathway_start_pos + (len(all_groups) - 1) * group_spacing / 2
            pathway_positions.append((pathway_center, pathway))
            
            current_pos += 1
        
        for bar in bar_data:
            color = get_color(bar['group'])
            ax.barh(bar['position'], bar['value'], color=color, alpha=0.8, 
                   edgecolor='black', linewidth=0.5, height=0.6)
        
        ax.set_yticks([pos for pos, _ in pathway_positions])
        ax.set_yticklabels([pathway for _, pathway in pathway_positions], 
                          fontsize=10, rotation=90, ha='right')
        
        separator_positions = []
        current_sep_pos = len(all_groups) * group_spacing - group_spacing/2
        for i in range(len(pathways) - 1):
            separator_positions.append(current_sep_pos + 0.5)
            current_sep_pos += len(all_groups) * group_spacing + 1
        
        for sep_pos in separator_positions:
            ax.axhline(y=sep_pos, color='gray', linestyle='--', alpha=0.7, linewidth=1)
        
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_title(f'{group_name}', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        ax.set_xlim(left=0)
        
        ax.set_ylim(-0.5, current_pos - 0.5)
    
    if isinstance(cmap, dict):
        legend_elements = [plt.Rectangle((0,0), 1, 1, facecolor=colors_dict.get(str(group), '#808080'), alpha=0.8, 
                                       edgecolor='black', linewidth=0.5) 
                          for group in all_groups]
    else:
        legend_elements = [plt.Rectangle((0,0), 1, 1, facecolor=colors(i), alpha=0.8, 
                                       edgecolor='black', linewidth=0.5) 
                          for i in range(len(all_groups))]
    
    fig.legend(legend_elements, all_groups, 
               loc='upper center', bbox_to_anchor=(0.5, 0.05), 
               ncol=len(all_groups), frameon=True, fontsize=10)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)  
    
    if save_file_name:
        plt.savefig(save_file_name, dpi=300, bbox_inches='tight')
        print(f"Figure saved as {save_file_name}")
    else:
        plt.show()

In [ ]:
pathway_barplot(all_pathway[all_pathway['group'] != "6"], pathway_dict, 
                 cmap = {
                            "1": '#ff7f0e',
                            "5": '#2ca02c',
                            "4": '#9467bd',
                            "8": '#bcbd22',
                        },
                save_file_name=f'{output_dir}/pathway_barplot.pdf'
                )

## sc-SVC subtypes analysis

### CAF_8

In [ ]:
CAF_8_cells_names = sc_SVC_adata[sc_SVC_adata.obs['SVC_cluster'] == "8",:].obs['cell_id'].values
tumor_adata_sp = adata_sp[adata_sp.obs['Level1'] == "Tumor" ,:]
CAF_8_adata_sp = adata_sp[adata_sp.obs['cell_id'].isin(CAF_8_cells_names),:]
adata = tumor_adata_sp.concatenate(CAF_8_adata_sp)  
adata.uns['Level1_colors'] = ['#bcbd22', '#f19493']

sc.pl.scatter(adata, x='x', y='y', color='Level1', size = 4)

### CAF_4

In [ ]:
import scanpy as sc

sc_SVC_adata = sc.read(f"{output_dir}/sc_SVC_spatial.h5ad")
CAF_adata_sc = sc.read(f"{output_dir}/sc_SVC_expr.h5ad")


In [ ]:
adata = adata_sp[adata_sp.obs['Level1'].isin(["Tumor", "Fibroblast"])]
caf_4_cell_ids = sc_SVC_adata.obs.loc[sc_SVC_adata.obs['SVC_cluster'] == "4", "cell_id"].values
adata.obs.loc[adata.obs['cell_id'].isin(caf_4_cell_ids), "Level1"] = "CAF_4"
sc.pl.scatter(adata, x="x", y="y", color="Level1")

In [ ]:
adata.obs['Level1'].value_counts()

In [ ]:
import scanpy as sc
import pandas as pd

tumor_indices = adata.obs[adata.obs['Level1'] == 'Tumor'].index
fibroblast_indices = adata.obs[adata.obs['Level1'] == 'Fibroblast'].index
caf4_indices = adata.obs[adata.obs['Level1'] == 'CAF_4'].index

tumor_sampled = np.random.choice(tumor_indices, size=20000, replace=False)
fibroblast_sampled = np.random.choice(fibroblast_indices, size=3000, replace=False)
caf4_sampled = caf4_indices 

sampled_indices = np.concatenate([tumor_sampled, fibroblast_sampled, caf4_sampled])

new_adata = adata[sampled_indices, :].copy()
new_order = ['Tumor', 'Fibroblast', 'CAF_4']
new_adata.obs['Level1'] = new_adata.obs['Level1'].cat.reorder_categories(new_order)

new_adata.uns['Level1_colors'] = [ '#f09094', '#d3d3d3', '#754e9e']
new_adata.obs['Level1'].value_counts()


In [ ]:
sc.pl.scatter(new_adata, x="x", y="y", color="Level1",size=10)

In [ ]:
# compare gene LPL expression across subtypes
import scanpy as sc

sc_SVC_adata = sc.read(f"{output_dir}/sc_SVC_spatial.h5ad")
CAF_adata_sc = sc.read(f"{output_dir}/sc_SVC_expr.h5ad")


cluster_nums = ['1','4','5','6', '8']
select_adata = CAF_adata_sc[CAF_adata_sc.obs['SVC_cluster'].isin(cluster_nums)]
sc.pp.normalize_per_cell(select_adata)


sc.pl.violin(select_adata, keys="LPL", groupby="SVC_cluster", show=False)
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42     
mpl.rcParams['ps.useafm'] = False     

plt.savefig(f"{output_dir}/sc_SVC_LPL_violin.pdf", dpi=300)

In [ ]:
def read_gmt(file_path):
    
    pathway_dict = {}
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0]
            genes = parts[2:]  # 跳过前两列（通路名称和描述）
            pathway_dict[pathway_name] = genes
    return pathway_dict

HYPOXIA_pathway = read_gmt("./pathway/HYPOXIA_related_pathways.gmt")
HYPOXIA_pathway.keys()


In [ ]:
import matplotlib.colors as mcolors
import numpy as np

colors = ["#d3d3d3", "#8b0000"]  
colors = ["white","#d3d3d3", "#8b0000"]  

n_bins = 100  

# colormap
cmap = mcolors.LinearSegmentedColormap.from_list("gray_to_darkred", colors, N=n_bins)

def plot_signature(adata, signatures, score_method = "score_genes", score_name = "TLS", size = None, color_map = cmap, plot_type = "scatter", groupby='SVC_cluster', max_score = None):
    adata = adata.copy()
    valid_genes = [gene for gene in signatures if gene in adata.var_names]
    print(len(valid_genes), len(signatures))
    if score_method == "score_genes":
        sc.tl.score_genes(
                    adata,
                    gene_list=valid_genes,
                    score_name=score_name,
                    use_raw=False,  
                )
    elif score_method == "AUC":
        import omicverse as ov
        ov.single.geneset_aucell(
                    adata=adata,
                    geneset_name=score_name,
                    geneset=valid_genes,
                    # AUC_threshold = 0.2
                )
        score_name = f"{score_name}_aucell"

    if max_score is not None:
        adata.obs[score_name] = np.clip(adata.obs[score_name], None, max_score)
        
    if plot_type == "scatter":
        sc.pl.scatter(adata, x="x", y="y", 
                    color=score_name, 
                    color_map=color_map,
                    size = size,
                    )
    else:
        sc.pl.violin(adata, score_name, groupby=groupby)
    return adata

In [ ]:
score_method="AUC"
score_method="score_genes"

size = 40
pathway_name = list(HYPOXIA_pathway.keys())[4]
print(pathway_name)
adata_sp_a = plot_signature(adata_sp[adata_sp.obs["Level1"] == "Tumor"], HYPOXIA_pathway[pathway_name], 
               score_method=score_method, score_name = pathway_name, 
               color_map = cmap,
               size = size,
               max_score=7)

### CAF_5

In [ ]:
import scanpy as sc

SMC_adata = adata_sc[adata_sc.obs['Level1']== "SMC",:]
SMC_adata.obs['SVC_cluster'] = "SMC"
CAF_adata_sc = sc.read(f"{output_dir}/sc_SVC_expr.h5ad")


cluster_nums = ['1','4','5','6', '8']
select_adata = CAF_adata_sc[CAF_adata_sc.obs['SVC_cluster'].isin(cluster_nums)]
select_adata = SMC_adata.concatenate(select_adata)
sc.pp.normalize_per_cell(select_adata)
# sc.pp.log1p(select_adata)
sc.pl.violin(select_adata, keys="MYH11", groupby="SVC_cluster")


In [ ]:
import pandas as pd
desired_order = ['SMC', '1', '4', '5', '6', '8']
select_adata.obs['SVC_cluster'] = pd.Categorical(select_adata.obs['SVC_cluster'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.violin(select_adata, keys="MYH11", groupby="SVC_cluster", rotation=90, show=False)

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42     
mpl.rcParams['ps.useafm'] = False     

plt.savefig(f"{output_dir}/sc_SVC_MYH11_violin.pdf", dpi=300)

In [ ]:
CAF_5_cells_names = sc_SVC_adata[sc_SVC_adata.obs['SVC_cluster'] == "5",:].obs['cell_id'].values
SMC_adata_sp = adata_sp[adata_sp.obs['Level1'] == "SMC" ,:]
CAF_5_adata_sp = adata_sp[adata_sp.obs['cell_id'].isin(CAF_5_cells_names),:]
adata = SMC_adata_sp.concatenate(CAF_5_adata_sp)  

In [ ]:

adata.uns['Level1_colors'] = ['#279865','#f7b678']

sc.pl.scatter(adata, x='x', y='y', color='Level1', size = 20)

## Pseudotime analysis 

In [ ]:
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import omicverse as ov
ov.plot_set()

adata=ov.pp.preprocess(adata, mode='shiftlog|pearson',n_HVGs=3000)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata,layer='scaled',n_pcs=50)

In [ ]:
Traj=ov.single.TrajInfer(adata,basis='X_umap',groupby='Level1',
                         use_rep='scaled|original|X_pca',n_comps=50)
Traj.set_origin_cells('SMC')
Traj.set_terminal_cells(["Fibroblast"])

In [ ]:
Traj.inference(method='palantir',num_waypoints=500)

In [ ]:
adata.obs['SVC_cluster'] = adata.obs['SVC_cluster'].astype(str)

adata.obs['SVC_cluster'] = (
    adata.obs['SVC_cluster']
         .astype('category')
         .cat.add_categories('SMC')
)

adata.obs['SVC_cluster'] = adata.obs['SVC_cluster'].fillna('SMC')

In [ ]:
adata.obs['SVC_cluster'] = adata.obs['SVC_cluster'].replace('nan', 'SMC')

In [ ]:
sc.pl.umap(adata, color=['palantir_pseudotime', 'MYH11','ACTA2', 'FAP', 'COL1A1'], cmap = "coolwarm")

In [ ]:
adata.uns['SVC_cluster_colors'] = ['#279865','#f7b678']
sc.pl.umap(adata, color=['SVC_cluster'], size=20)